In [28]:
!pip install joblib

In [29]:
# Import Libraries
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef,
    roc_auc_score
)

In [30]:
# Load Dataset
df = pd.read_csv(
    "Dry_Bean_Dataset.csv"
)

print(df.shape)

df.head()

(13611, 17)


,Area,Perimeter,MajorAxisLength,MinorAxisLength,AspectRation,Eccentricity,ConvexArea,EquivDiameter,Extent,Solidity,roundness,Compactness,ShapeFactor1,ShapeFactor2,ShapeFactor3,ShapeFactor4,Class
0,28395,610.291,208.178117,173.888747,1.197191,0.549812,28715,190.141097,0.763923,0.988856,0.958027,0.913358,0.007332,0.003147,0.834222,0.998724,SEKER
1,28734,638.018,200.524796,182.734419,1.097356,0.411785,29172,191.272751,0.783968,0.984986,0.887034,0.953861,0.006979,0.003564,0.909851,0.998430,SEKER
2,29380,624.110,212.826130,175.931143,1.209713,0.562727,29690,193.410904,0.778113,0.989559,0.947849,0.908774,0.007244,0.003048,0.825871,0.999066,SEKER
3,30008,645.884,210.557999,182.516516,1.153638,0.498616,30724,195.467062,0.782681,0.976696,0.903936,0.928329,0.007017,0.003215,0.861794,0.994199,SEKER
4,30140,620.134,201.847882,190.279279,1.060798,0.333680,30417,195.896503,0.773098,0.990893,0.984877,0.970516,0.006697,0.003665,0.941900,0.999166,SEKER


In [31]:
# Checking Columns
print(df.columns)

Index(['Area', 'Perimeter', 'MajorAxisLength', 'MinorAxisLength',
       'AspectRation', 'Eccentricity', 'ConvexArea', 'EquivDiameter', 'Extent',
       'Solidity', 'roundness', 'Compactness', 'ShapeFactor1', 'ShapeFactor2',
       'ShapeFactor3', 'ShapeFactor4', 'Class'],
      dtype='object')


In [32]:
# Separate Features and Target
X = df.drop(
    "Class",
    axis=1
)

y = df["Class"]

In [33]:
# Encode Labels
encoder = LabelEncoder()

y = encoder.fit_transform(y)

In [34]:
# Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [35]:
# Save Test Data
test_df = X_test.copy()

test_df["Class"] = y_test

test_df.to_csv(
    "test_data.csv",
    index=False
)

print(test_df.shape)

(2723, 17)


In [36]:
# Scale Features
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(
    X_train
)

X_test_scaled = scaler.transform(
    X_test
)

In [37]:
# Save Scaler
joblib.dump(
    scaler,
    "scaler.pkl"
)

['scaler.pkl']

In [38]:
# Define Models
models = {

    "Logistic Regression":
    LogisticRegression(max_iter=3000),

    "Decision Tree":
    DecisionTreeClassifier(random_state=42),

    "KNN":
    KNeighborsClassifier(),

    "Naive Bayes":
    GaussianNB(),

    "SVM":
    SVC(probability=True)
}

In [39]:
# Train Models + Calculate Metrics
results = []

for name, model in models.items():

    if name in [
        "Logistic Regression",
        "KNN",
        "SVM"
    ]:

        model.fit(
            X_train_scaled,
            y_train
        )

        preds = model.predict(
            X_test_scaled
        )

        probs = model.predict_proba(
            X_test_scaled
        )

    else:

        model.fit(
            X_train,
            y_train
        )

        preds = model.predict(
            X_test
        )

        probs = model.predict_proba(
            X_test
        )

    accuracy = accuracy_score(
        y_test,
        preds
    )

    precision = precision_score(
        y_test,
        preds,
        average="weighted"
    )

    recall = recall_score(
        y_test,
        preds,
        average="weighted"
    )

    f1 = f1_score(
        y_test,
        preds,
        average="weighted"
    )

    auc = roc_auc_score(
        y_test,
        probs,
        multi_class="ovr"
    )

    mcc = matthews_corrcoef(
        y_test,
        preds
    )

    results.append([
        name,
        accuracy,
        auc,
        precision,
        recall,
        f1,
        mcc
    ])

    filename = (
        name.lower()
        .replace(" ","_")
        + ".pkl"
    )

    joblib.dump(
        model,
        filename
    )

    print(name, "saved")

Logistic Regression saved
Decision Tree saved
KNN saved
Naive Bayes saved
SVM saved


In [40]:
# Create Metrics Table
metrics_df = pd.DataFrame(
    results,
    columns=[
        "Model",
        "Accuracy",
        "AUC",
        "Precision",
        "Recall",
        "F1",
        "MCC"
    ]
)

metrics_df

,Model,Accuracy,AUC,Precision,Recall,F1,MCC
0,Logistic Regression,0.920676,0.994768,0.921533,0.920676,0.920873,0.904167
1,Decision Tree,0.892031,0.944996,0.891693,0.892031,0.891630,0.869569
2,KNN,0.916636,0.983317,0.917409,0.916636,0.916805,0.899204
3,Naive Bayes,0.763863,0.967193,0.765371,0.763863,0.761474,0.715406
4,SVM,0.922145,0.994931,0.922426,0.922145,0.922177,0.905870


In [41]:
# Save Metrics
metrics_df.to_csv(
    "metrics.csv",
    index=False
)

In [42]:
# Download Files
from google.colab import files

files.download("metrics.csv")
files.download("test_data.csv")
files.download("scaler.pkl")

files.download("logistic_regression.pkl")
files.download("decision_tree.pkl")
files.download("knn.pkl")
files.download("naive_bayes.pkl")
files.download("svm.pkl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

... (6 outputs left out)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>